In [13]:
from langgraph.graph import StateGraph, START, END
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from typing import TypedDict, Annotated
from pydantic import BaseModel, Field  
import os
import operator


In [3]:
load_dotenv()

True

In [4]:
model = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=os.getenv("GROQ_API_KEY")
)

In [5]:
class EvaluationSchema(BaseModel):
    feedback: str = Field(description="Detailed feedback on the essay")
    score: int = Field(description="Score from 1 to 10 evaluating the essay", ge=1, le=10)

In [6]:
structured_model = model.with_structured_output(EvaluationSchema)

In [7]:
essay = """Artificial Intelligence (AI) has rapidly transformed various industries, from healthcare to finance. Its ability to analyze vast amounts of data and make predictions has led to significant advancements. However, the ethical implications of AI, such as bias and privacy concerns, must be addressed to ensure its responsible use. As we continue to integrate AI into our daily lives, it is crucial to establish guidelines that promote transparency and accountability in AI systems."""

In [8]:
prompt= f"""
You are an expert essay reviewer. Please provide detailed feedback on the following essay and assign it a score from 1 to 10.\n {essay}"""

In [11]:
structured_model.invoke(prompt).score

8

In [14]:
class EssayState(TypedDict):
    essay: str
    lang_feedback: str
    analysis_feedback: int
    clarity_feedback: str
    overall_feedback: str

    individual_scores: Annotated[list[int], operator.add]
    avg_score: float



In [15]:
def evaluate_language(state: EssayState) -> EssayState:
    prompt = f"""
        You are an expert essay reviewer. Please Evlauate the language of the following essay, provide detailed feedback  and assign it a score from 1 to 10.\n {state['essay']}"""
    output = structured_model.invoke(prompt)
    
    return {'lang_feedback': output.feedback, 'individual_scores': [output.score]}


In [16]:
def evaluate_analysis(state: EssayState) -> EssayState:
    prompt = f"""
        You are an expert essay reviewer. Please evaluate the the depth of analysis of the following essay, provide detailed feedback  and assign it a score from 1 to 10.\n {state['essay']}"""
    output = structured_model.invoke(prompt)
    
    return {'individual_feedback': output.feedback, 'individual_scores': [output.score]}


In [17]:
def evaluate_thought(state: EssayState) -> EssayState:
    prompt = f"""
        You are an expert essay reviewer. Please evaluate clarity of thought of the following essay, provide detailed feedback  and assign it a score from 1 to 10.\n {state['essay']}"""
    output = structured_model.invoke(prompt)
    
    return {'clarity_feedback': output.feedback, 'individual_scores': [output.score]}

In [18]:
def final_evaluation(state: EssayState) -> EssayState:
    #summary feedback
    prompt = """ based on th following feedbacks, provide an overall feedback summary:
    Language Feedback: {state['lang_feedback']}
    Analysis Feedback: {state['analysis_feedback']}
    Clarity Feedback: {state['clarity_feedback']}"""
    overall_feedback =model.invoke(prompt).content
    #average score
    average_score = sum(state['individual_scores']) / len(state['individual_scores'])
    return {'overall_feedback': overall_feedback, 'avg_score': average_score}

In [19]:
graph = StateGraph(EssayState)

graph.add_node('evaluate_language', evaluate_language)
graph.add_node('evaluate_analysis', evaluate_analysis)
graph.add_node('evaluate_thought', evaluate_thought)
graph.add_node('final_evaluation', final_evaluation)

In [ ]:
#edges
graph.add_edge(START, 'evaluate_language')
graph.add_edge(START, 'evaluate_analysis')   
graph.add_edge(START, 'evaluate_thought')

graph.add_edge('evaluate_language', 'final_evaluation')
graph.add_edge('evaluate_analysis', 'final_evaluation')
graph.add_edge('evaluate_thought', 'final_evaluation')

graph.add_edge('final_evaluation', END)